In [0]:
# Fetch GDP growth estimates for all OECD countries using World Bank API
import pandas as pd
import requests
from datetime import datetime

# OECD member countries (38 members as of 2023)
oecd_countries = {
    'AUS': 'Australia', 'AUT': 'Austria', 'BEL': 'Belgium', 'CAN': 'Canada',
    'CHL': 'Chile', 'COL': 'Colombia', 'CRI': 'Costa Rica', 'CZE': 'Czech Republic',
    'DNK': 'Denmark', 'EST': 'Estonia', 'FIN': 'Finland', 'FRA': 'France',
    'DEU': 'Germany', 'GRC': 'Greece', 'HUN': 'Hungary', 'ISL': 'Iceland',
    'IRL': 'Ireland', 'ISR': 'Israel', 'ITA': 'Italy', 'JPN': 'Japan',
    'KOR': 'South Korea', 'LVA': 'Latvia', 'LTU': 'Lithuania', 'LUX': 'Luxembourg',
    'MEX': 'Mexico', 'NLD': 'Netherlands', 'NZL': 'New Zealand', 'NOR': 'Norway',
    'POL': 'Poland', 'PRT': 'Portugal', 'SVK': 'Slovakia', 'SVN': 'Slovenia',
    'ESP': 'Spain', 'SWE': 'Sweden', 'CHE': 'Switzerland', 'TUR': 'Turkey',
    'GBR': 'United Kingdom', 'USA': 'United States'
}

print(f"Fetching GDP growth data for {len(oecd_countries)} OECD countries...\n")

# World Bank API endpoint for GDP growth (annual %)
# Indicator: NY.GDP.MKTP.KD.ZG - GDP growth (annual %)
country_codes = ';'.join(oecd_countries.keys())
start_year = 2020
end_year = 2026  # Include forecast years if available

url = f"https://api.worldbank.org/v2/country/{country_codes}/indicator/NY.GDP.MKTP.KD.ZG"
params = {
    'date': f'{start_year}:{end_year}',
    'format': 'json',
    'per_page': 500
}

try:
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    if len(data) > 1 and data[1]:
        # Parse the data
        records = []
        for item in data[1]:
            if item['value'] is not None:
                records.append({
                    'Country Code': item['country']['id'],
                    'Country': item['country']['value'],
                    'Year': item['date'],
                    'GDP Growth (%)': round(item['value'], 2)
                })
        
        # Create DataFrame
        df = pd.DataFrame(records)
        df = df.sort_values(['Year', 'Country'], ascending=[False, True])
        
        print(f"✓ Successfully fetched {len(df)} data points\n")
        
        # Show most recent year's data
        latest_year = df['Year'].max()
        latest_data = df[df['Year'] == latest_year].sort_values('GDP Growth (%)', ascending=False)
        
        print(f"GDP Growth Rates for {latest_year}:")
        print("="*60)
        display(latest_data)
        
        # Summary statistics
        print(f"\nSummary Statistics for {latest_year}:")
        print(f"  Average GDP Growth: {latest_data['GDP Growth (%)'].mean():.2f}%")
        print(f"  Median GDP Growth: {latest_data['GDP Growth (%)'].median():.2f}%")
        print(f"  Highest: {latest_data.iloc[0]['Country']} ({latest_data.iloc[0]['GDP Growth (%)']}%)")
        print(f"  Lowest: {latest_data.iloc[-1]['Country']} ({latest_data.iloc[-1]['GDP Growth (%)']}%)")
        
        # Show trend for last few years
        print(f"\nGDP Growth Trend (All Years):")
        display(df.pivot_table(index='Country', columns='Year', values='GDP Growth (%)'))
        
    else:
        print("No data returned from World Bank API")
        
except Exception as e:
    print(f"Error: {e}")
    print(f"\nOECD Countries ({len(oecd_countries)}):")
    for code, name in oecd_countries.items():
        print(f"  {code}: {name}")

Fetching GDP growth data for 38 OECD countries...

✓ Successfully fetched 228 data points

GDP Growth Rates for 2025:


Country Code,Country,Year,GDP Growth (%)
IE,Ireland,2025,12.34
CR,Costa Rica,2025,4.56
TR,Turkiye,2025,3.6
PL,Poland,2025,3.57
IL,Israel,2025,2.93
DK,Denmark,2025,2.93
LT,Lithuania,2025,2.92
ES,Spain,2025,2.82
CO,Colombia,2025,2.64
CZ,Czechia,2025,2.58



Summary Statistics for 2025:
  Average GDP Growth: 1.93%
  Median GDP Growth: 1.37%
  Highest: Ireland (12.34%)
  Lowest: Finland (0.17%)

GDP Growth Trend (All Years):


2020,2021,2022,2023,2024,2025
-0.13,2.01,4.25,3.58,1.37,1.35
-6.32,4.92,5.33,-0.79,-0.66,0.62
-4.79,6.25,4.02,1.56,1.1,0.98
-5.04,5.95,4.7,1.95,2.05,1.74
-6.14,11.34,2.06,0.68,2.81,2.46
-7.19,10.8,7.33,0.84,1.49,2.64
-4.14,8.22,5.46,4.79,4.08,4.56
-5.3,4.03,2.85,0.05,1.25,2.58
-1.78,6.5,0.44,0.6,3.48,2.93
-2.88,8.25,-1.23,-2.74,-0.09,0.58


In [0]:
# Calculate expected inflation for Mexico in 2026 using Taylor Rule
import pandas as pd
import requests
import numpy as np

print("Taylor Rule Inflation Forecast for Mexico (2026)")
print("="*60)

# Taylor Rule: i = r* + π + 0.5(π - π*) + 0.5(y_gap)
# Rearranged to solve for inflation: π = (i - r* - 0.5*π* - 0.5*y_gap) / 1.5

# Parameters for Mexico
target_inflation = 3.0  # Banco de México's inflation target (%)
real_equilibrium_rate = 2.0  # Long-term real interest rate (%)

print(f"\nTaylor Rule Parameters:")
print(f"  Target Inflation (π*): {target_inflation}%")
print(f"  Real Equilibrium Rate (r*): {real_equilibrium_rate}%")

# Get Mexico's current policy rate from World Bank or IMF
try:
    # Fetch Mexico's policy rate (Real interest rate)
    url = "https://api.worldbank.org/v2/country/MEX/indicator/FR.INR.RINR"
    params = {'date': '2020:2026', 'format': 'json', 'per_page': 50}
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    rate_data = response.json()
    
    if len(rate_data) > 1 and rate_data[1]:
        rates = [(item['date'], item['value']) for item in rate_data[1] if item['value'] is not None]
        latest_rate_year, latest_rate = rates[0] if rates else ('2024', 8.0)
        policy_rate = latest_rate
    else:
        # Use typical Banco de México rate if API fails
        policy_rate = 10.5  # Approximate current rate
        latest_rate_year = '2025'
    
    print(f"  Policy Interest Rate ({latest_rate_year}): {policy_rate}%")
    
except Exception as e:
    print(f"  Note: Using estimated policy rate (API error: {e})")
    policy_rate = 10.5  # Conservative estimate for Banco de México rate
    print(f"  Estimated Policy Rate: {policy_rate}%")

# Calculate output gap from GDP growth data (already in df)
mex_gdp = df[df['Country Code'] == 'MX'].sort_values('Year')

if not mex_gdp.empty:
    print(f"\nMexico GDP Growth History:")
    for _, row in mex_gdp.iterrows():
        print(f"  {row['Year']}: {row['GDP Growth (%)']}%")
    
    # Calculate potential GDP growth (average of recent years)
    recent_growth = mex_gdp[mex_gdp['Year'].astype(int) >= 2020]['GDP Growth (%)'].mean()
    potential_growth = 2.5  # Mexico's long-term potential growth rate
    
    # Output gap = Actual growth - Potential growth
    gdp_2025 = mex_gdp[mex_gdp['Year'] == '2025']['GDP Growth (%)'].values
    actual_growth_2025 = gdp_2025[0] if len(gdp_2025) > 0 else recent_growth
    output_gap = actual_growth_2025 - potential_growth
    
    print(f"\nOutput Gap Analysis:")
    print(f"  Actual Growth (2025): {actual_growth_2025}%")
    print(f"  Potential Growth: {potential_growth}%")
    print(f"  Output Gap: {output_gap:.2f}%")
else:
    output_gap = -1.0  # Assume slight negative output gap
    print(f"\nAssumed Output Gap: {output_gap}%")

# Apply Taylor Rule to estimate implied inflation
# Standard Taylor Rule: i = r* + π + 0.5(π - π*) + 0.5(y_gap)
# Solving for π: i = r* + π + 0.5π - 0.5π* + 0.5y_gap
#                 i = r* + 1.5π - 0.5π* + 0.5y_gap
#                 π = (i - r* + 0.5π* - 0.5y_gap) / 1.5

implied_inflation = (policy_rate - real_equilibrium_rate + 0.5*target_inflation - 0.5*output_gap) / 1.5

print(f"\n" + "="*60)
print(f"TAYLOR RULE RESULT")
print(f"="*60)
print(f"Expected Inflation for Mexico (2026): {implied_inflation:.2f}%")
print(f"\nInterpretation:")

if implied_inflation > target_inflation:
    diff = implied_inflation - target_inflation
    print(f"  • Inflation is expected to be {diff:.2f} percentage points")
    print(f"    ABOVE the {target_inflation}% target")
    print(f"  • This suggests Banco de México may maintain tight monetary policy")
elif implied_inflation < target_inflation:
    diff = target_inflation - implied_inflation
    print(f"  • Inflation is expected to be {diff:.2f} percentage points")
    print(f"    BELOW the {target_inflation}% target")
    print(f"  • This suggests room for monetary easing")
else:
    print(f"  • Inflation is at target ({target_inflation}%)")
    print(f"  • Monetary policy appears well-calibrated")

print(f"\nCaveats:")
print(f"  • Taylor Rule is a guideline, not a precise forecast")
print(f"  • Actual inflation depends on many other factors")
print(f"  • Exchange rate movements and external shocks matter")
print(f"  • Banco de México's actual target range is 2-4%")

Taylor Rule Inflation Forecast for Mexico (2026)

Taylor Rule Parameters:
  Target Inflation (π*): 3.0%
  Real Equilibrium Rate (r*): 2.0%
  Policy Interest Rate (2025): 3.9709808535188%

Mexico GDP Growth History:
  2020: -8.35%
  2021: 6.05%
  2022: 3.71%
  2023: 3.11%
  2024: 1.35%
  2025: 0.56%

Output Gap Analysis:
  Actual Growth (2025): 0.56%
  Potential Growth: 2.5%
  Output Gap: -1.94%

TAYLOR RULE RESULT
Expected Inflation for Mexico (2026): 2.96%

Interpretation:
  • Inflation is expected to be 0.04 percentage points
    BELOW the 3.0% target
  • This suggests room for monetary easing

Caveats:
  • Taylor Rule is a guideline, not a precise forecast
  • Actual inflation depends on many other factors
  • Exchange rate movements and external shocks matter
  • Banco de México's actual target range is 2-4%
